# chapterwise PDF Extract on Kaggle

**Dataset:** attach **`chapterwise-models`** (latest version).

**Publish updates from laptop** (when code changes):
```bash
backend\kaggle\publish_chapterwise_dataset.bat
```
Requires `KAGGLE_USERNAME` + `KAGGLE_API_TOKEN` in `backend/.env`.

**Each session:** GPU T4 x2 · Internet ON · set `KAGGLE_API_SECRET` in cell 2 · copy tunnel URL to laptop `backend/.env`.

Stop the notebook when done — GPU hours tick while the session is open.

In [ ]:
# Copy server modules from the attached chapterwise-models dataset
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

REQUIRED = (
    "chapterwise_gpu_server.py",
    "llm_server.py",
    "security.py",
    "pdf_extract.py",
)

missing = []
for name in REQUIRED:
    copied = False
    for src in INPUT.rglob(name):
        shutil.copy2(src, WORK / name)
        print(f"Copied {name} <- {src}")
        copied = True
        break
    if not copied:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Missing from dataset: " + ", ".join(missing)
        + "\nRe-run publish_chapterwise_dataset.bat on your laptop,"
        + " attach the new dataset version, then re-run this cell."
    )

print("\nWorking dir:", sorted(p.name for p in WORK.iterdir() if p.is_file()))

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Kaggle API secret — Add-ons -> Secrets, label KAGGLE_API_SECRET (16+ chars; must match laptop .env)
try:
    kaggle_secret = user_secrets.get_secret("KAGGLE_API_SECRET")
except Exception as exc:
    kaggle_secret = ""
    print(f"KAGGLE_API_SECRET secret lookup failed: {exc}")
print(
    f"KAGGLE_API_SECRET retrieved: {bool(kaggle_secret and kaggle_secret.strip())}, "
    f"len={len(kaggle_secret or '')}"
)
if kaggle_secret and kaggle_secret.strip():
    os.environ["KAGGLE_API_SECRET"] = kaggle_secret.strip()
else:
    print("WARNING: KAGGLE_API_SECRET missing — laptop /chat/completions will 401")

# Hugging Face token for gated Llama — Add-ons -> Secrets, label HF_TOKEN
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as exc:
    hf_token = ""
    print(f"HF_TOKEN secret lookup failed: {exc}")
# TEMP diagnostic — must appear BEFORE Llama load (server starts later via !python)
print(
    f"HF_TOKEN present: {bool(hf_token)}, length: {len(hf_token) if hf_token else 0}"
)
if hf_token and hf_token.strip():
    os.environ["HF_TOKEN"] = hf_token.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token.strip()
else:
    print("WARNING: HF_TOKEN missing or empty — Llama load will 401 on gated repo")

os.environ.setdefault("CHAPTERWISE_LLM_MODEL", "meta-llama/Llama-3.2-1B-Instruct")
os.environ.setdefault("CHAPTERWISE_LLM_PRELOAD", "1")
os.environ["CHAPTERWISE_LLM_MAX_NEW_TOKENS"] = "1024"
os.environ["CHAPTERWISE_LLM_MAX_INPUT_CHARS"] = "12000"
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
# Speed tuning (defaults also set inside chapterwise_gpu_server.py)
os.environ.setdefault("CHAPTERWISE_EXTRACT_WORKERS", str(os.cpu_count() or 4))
os.environ.setdefault("CHAPTERWISE_DEFER_IMAGE_B64", "1")
os.environ.setdefault("CHAPTERWISE_SKIP_OVERLAP", "1")
os.environ.setdefault("CHAPTERWISE_USE_PROCESS_POOL", "1")

!python /kaggle/working/chapterwise_gpu_server.py